# Driver monitoring: local data, optional training

Run the numbered cells in order. The default checks your local prepared data and saved subject splits; **it does not train, download data, install packages, or change any Kaggle session**.

Choose either **State Farm distraction** or **RT-BENE eye state**. These are separate per-frame classifiers. Gaze off road, yawning and head nodding need suitable labels and are not trained here. DMD stays held out for later testing and never enters training.

Your notebook and source code live in this repository. A remote kernel runs on its own machine: it needs an explicit copy of both source code and prepared data. See `docs/LOCAL_NOTEBOOK.md` for preparation and manual transfer.


## 1. Choose the task and check where Python is running

Leave `RUN_TRAINING = False` for inspection. Selecting a remote kernel does not move files from your laptop. For Kaggle, attach a readable source snapshot and a prepared-data bundle first, then fill in the two remote paths below. The prepared bundle must contain `data/state_farm/prepared`, `data/rt_bene/prepared` (as needed), and `data/splits`.

`FAST_RUN` is only a small training smoke run when training is enabled; it is not a publishable evaluation.


In [ ]:
from pathlib import Path
import importlib.metadata
import importlib.util
import json
import os
import platform
import subprocess
import sys

RUN_TRAINING = False
TASK = "state_farm"  # "state_farm" or "rt_bene"
FAST_RUN = False
DEVICE = "cpu"  # Set to "cuda" only with a GPU kernel.
LOCAL_REPO_ROOT = ""  # Usually blank: find this repo from the notebook directory.
REMOTE_REPO_ROOT = ""  # Example: /kaggle/input/my-source/project
REMOTE_BUNDLE_ROOT = ""  # Example: /kaggle/input/my-prepared-data
RESULTS_RUN = ""  # Optional existing experiment folder to inspect in step 6.

ON_KAGGLE = Path("/kaggle/working").is_dir()
print("Kernel:", "Kaggle remote" if ON_KAGGLE else "local / other kernel")
print("Python:", sys.executable)
print("Version:", platform.python_version())
print("Working directory:", Path.cwd())
print("Task:", TASK, "| Training enabled:", RUN_TRAINING, "| Device:", DEVICE)
assert TASK in {"state_farm", "rt_bene"}, "Choose one of the two supported tasks."
assert DEVICE in {"cpu", "cuda"}
if ON_KAGGLE:
    print("This kernel cannot read your laptop's E: drive. Remote copies are required.")


## 2. Locate code and prepared data

This only reads paths and checks installed packages. It never clones a repository or downloads raw data. Local setup instructions are in the companion guide. On a different remote service, explicitly set `LOCAL_REPO_ROOT` to the source on that machine; automatic Kaggle detection only applies to `/kaggle` kernels.


In [ ]:
if ON_KAGGLE:
    REPO = Path(REMOTE_REPO_ROOT).resolve() if REMOTE_REPO_ROOT else None
    BUNDLE = Path(REMOTE_BUNDLE_ROOT).resolve() if REMOTE_BUNDLE_ROOT else None
else:
    candidates = (
        [Path(LOCAL_REPO_ROOT).resolve()] if LOCAL_REPO_ROOT else [Path.cwd(), *Path.cwd().parents]
    )
    REPO = next((p for p in candidates if (p / "src/train/phase1.py").is_file()), None)
    BUNDLE = REPO

REPO_READY = REPO is not None and (REPO / "src/train/phase1.py").is_file()
DATA_BASE = BUNDLE / "data" if BUNDLE else None
SPECS = {
    "state_farm": {
        "config": "configs/state_farm_pretrained.yaml",
        "label": "label_distraction",
        "classes": [0, 1, 2, 3, 5],
    },
    "rt_bene": {"config": "configs/phase1.yaml", "label": "label_drowsiness", "classes": [0, 1]},
}
TRAIN_PACKAGES = {
    "torch": "torch",
    "torchvision": "torchvision",
    "numpy": "numpy",
    "PIL": "Pillow",
    "sklearn": "scikit-learn",
    "yaml": "PyYAML",
}
missing_packages = [
    name for module, name in TRAIN_PACKAGES.items() if importlib.util.find_spec(module) is None
]
print("Source root:", REPO or "MISSING: set a source path")
print("Prepared bundle root:", BUNDLE or "MISSING: set a bundle path")
print("Missing training packages:", missing_packages or "none")
if REPO_READY:
    # Prefer this explicit source snapshot over any editable installation in the kernel.
    if str(REPO) in sys.path:
        sys.path.remove(str(REPO))
    sys.path.insert(0, str(REPO))
    loaded_src = sys.modules.get("src")
    if loaded_src is not None and Path(loaded_src.__file__).resolve() != REPO / "src/__init__.py":
        raise RuntimeError(
            "Another source snapshot is already imported. Restart the kernel, then rerun."
        )
else:
    print("BLOCKED: source code is not available in this kernel.")
if DATA_BASE:
    for task in SPECS:
        print(task, "prepared root:", DATA_BASE / task / "prepared")
        print(task, "saved split:", DATA_BASE / "splits" / f"{task}_splits.json")


## 3. Check manifests, every image path, and fixed subject splits

Preparation happens explicitly in the terminal (`python scripts/prepare_local_data.py`; see the guide). This cell reads each prepared `manifest.csv`, completed preparation report and saved split JSON. Missing data are reported as blocked. Existing splits must match the manifest, seed and fractions exactly; this cell never creates or reshuffles them. All images are checked for existence; this is not a full image-decode audit.

RT-BENE uses labeled left-eye crops: **0 = open, 1 = closed/blinking**. State Farm uses the existing five project classes. Class mappings retain the limitations described in the dataset documentation.


In [ ]:
from collections import Counter

readiness = {}
loaded = {}
if REPO_READY and importlib.util.find_spec("yaml") is not None:
    import yaml
    from src.data.manifest import load_manifest
    from src.data.splits import assert_no_leakage, fingerprint
    import src.data.manifest as manifest_module

    assert Path(manifest_module.__file__).resolve() == REPO / "src/data/manifest.py"

    for task, spec in SPECS.items():
        try:
            if DATA_BASE is None:
                raise FileNotFoundError("Prepared bundle root is not set")
            root = (DATA_BASE / task / "prepared").resolve()
            manifest = root / "manifest.csv"
            split_path = DATA_BASE / "splits" / f"{task}_splits.json"
            config_path = REPO / spec["config"]
            config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
            assert config.get("label_key", "label_drowsiness") == spec["label"]
            assert config["class_ids"] == spec["classes"], "Unexpected task class configuration"
            rows = load_manifest(manifest)
            report_path = DATA_BASE / task / "preparation_report.json"
            preparation = json.loads(report_path.read_text(encoding="utf-8"))
            assert preparation["dataset"] == task, "Preparation report dataset differs"
            assert preparation["images_decoded"] == len(rows), "Preparation decode count differs"
            assert preparation["missing_images"] == 0, "Preparation reported missing images"
            assert preparation["corrupt_images"] == 0, "Preparation reported corrupt images"
            assert (
                preparation["split_provenance"][
                    "exact_sample_labels_and_subject_memberships_preserved"
                ]
                is True
            ), "Historical split preservation not verified"
            assert {r["dataset"] for r in rows} == {task}, "Unexpected dataset; DMD is forbidden"
            missing_images = []
            for row in rows:
                image = (root / row["path"]).resolve()
                if not image.is_relative_to(root):
                    raise ValueError("Image path escapes the prepared root")
                if not image.is_file():
                    missing_images.append(row["path"])
            if missing_images:
                raise FileNotFoundError(
                    f"{len(missing_images)} images missing; first: {missing_images[0]}"
                )
            saved = json.loads(split_path.read_text(encoding="utf-8"))
            assert saved["fingerprint"] == fingerprint(rows), "Manifest differs from saved split"
            assert saved["seed"] == config["seed"], "Saved split seed differs from config"
            assert saved["fractions"] == config["split_fractions"], "Split fractions differ"
            parts = saved["splits"]
            assert set(parts) == {"train", "val", "test"}
            assert_no_leakage(parts)
            assert fingerprint([r for group in parts.values() for r in group]) == fingerprint(
                rows
            ), "Saved split rows differ"
            for name, group in parts.items():
                support = {r[spec["label"]] for r in group} - {-1}
                assert group and support, f"Empty or unlabeled {name} split"
                assert support <= set(spec["classes"]), f"Unsupported labels in {name}"
                if name in {"train", "val"}:
                    assert support == set(spec["classes"]), f"Missing class support in {name}"
            loaded[task] = {
                "root": root,
                "manifest": manifest,
                "split_path": split_path,
                "config_path": config_path,
                "config": config,
                "parts": parts,
                "fingerprint": saved["fingerprint"],
            }
            readiness[task] = {
                "ready": True,
                "images": len(rows),
                "subjects": len({r["subject_id"] for r in rows}),
                "split_fingerprint": saved["fingerprint"],
                "splits": {
                    name: {
                        "images": len(group),
                        "subjects": len({r["subject_id"] for r in group}),
                        "labels": dict(Counter(r[spec["label"]] for r in group)),
                    }
                    for name, group in parts.items()
                },
            }
        except Exception as exc:
            readiness[task] = {"ready": False, "reason": f"{type(exc).__name__}: {exc}"}
else:
    readiness = {
        task: {"ready": False, "reason": "Source root or PyYAML missing (step 2)"} for task in SPECS
    }
print(json.dumps(readiness, indent=2))
print("Selected task:", "READY" if readiness[TASK]["ready"] else "BLOCKED")


## 4. Read the training plan and source identity

No experiment folder is created until training is enabled. State Farm uses the existing pretrained MobileNet configuration (the first training run may need the official torchvision weights downloaded or already cached). The RT-BENE baseline starts without pretrained weights. Training code is shared with the repository, not duplicated in this notebook.

The source manifest below hashes the visible Python source and configs. Each experiment records these hashes, the Git revision when available, the effective config and input/split fingerprints. Dirty local changes are therefore visible even when the Git revision is unchanged.


In [ ]:
import hashlib


def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


source_files = {}
source_revision = None
source_dirty = None
if REPO_READY:
    paths = sorted(
        set(REPO.glob("src/**/*.py")) | set(REPO.glob("configs/*.yaml")) | {REPO / "pyproject.toml"}
    )
    source_files = {p.relative_to(REPO).as_posix(): sha256_file(p) for p in paths if p.is_file()}
    try:
        source_revision = subprocess.check_output(
            ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True, stderr=subprocess.DEVNULL
        ).strip()
        source_dirty = bool(
            subprocess.check_output(
                ["git", "-C", str(REPO), "status", "--porcelain", "--untracked-files=no"],
                text=True,
                stderr=subprocess.DEVNULL,
            ).strip()
        )
    except (OSError, subprocess.CalledProcessError):
        marker = REPO / "SOURCE_COMMIT"
        source_revision = (
            marker.read_text(encoding="utf-8").strip()
            if marker.is_file()
            else "snapshot-without-git"
        )
    print("Source revision:", source_revision, "| Tracked changes:", source_dirty)
    print("Source/config files hashed:", len(source_files))

if readiness[TASK]["ready"]:
    selected = loaded[TASK]
    print("Config:", selected["config_path"])
    print(json.dumps(selected["config"], indent=2))
    print("Fixed split:", selected["split_path"])
    print("Device:", DEVICE, "| Fast smoke run:", FAST_RUN)
    print(
        "Outputs parent:",
        Path("/kaggle/working/experiments") if ON_KAGGLE else REPO / "results/experiments",
    )
else:
    print("Resolve the selected task's blocked status before enabling training.")


## 5. Optional training

To train intentionally, change `RUN_TRAINING` in step 1, choose the task/device, then rerun steps 1–5. Local CPU training may take a long time. GPU training requires an available CUDA runtime in the selected kernel.

Every invocation creates a new timestamped experiment folder. Existing experiments are never resumed or overwritten. The shared trainer also rejects nonempty model output folders. No DMD, gaze, yawn or nod training is offered here. A failed run keeps its log and provenance for inspection.


In [ ]:
from datetime import datetime, timezone
import uuid

if not RUN_TRAINING:
    print("Inspection only. Training skipped; no experiment folder created.")
else:
    assert readiness[TASK]["ready"], readiness[TASK]
    assert not missing_packages, missing_packages
    selected = loaded[TASK]
    import torch
    import torchvision

    assert Path(__import__("src").__file__).resolve() == REPO / "src/__init__.py"
    if DEVICE == "cuda":
        assert torch.cuda.is_available(), (
            "CUDA unavailable: choose a GPU kernel or set DEVICE='cpu'."
        )
    parent = Path("/kaggle/working/experiments") if ON_KAGGLE else REPO / "results/experiments"
    run_id = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        + "_"
        + TASK
        + "_"
        + uuid.uuid4().hex[:8]
    )
    experiment = parent / run_id
    experiment.mkdir(parents=True, exist_ok=False)
    model_output = experiment / "model"
    # Preserve the exact effective config; no source file or SOURCE_COMMIT is modified.
    effective_config = experiment / "config.yaml"
    effective_config.write_text(yaml.safe_dump(selected["config"]), encoding="utf-8")
    command = [
        sys.executable,
        "-m",
        "src.train.phase1",
        "--data-root",
        str(selected["root"]),
        "--manifest",
        str(selected["manifest"]),
        "--config",
        str(effective_config),
        "--output",
        str(model_output),
        "--splits",
        str(selected["split_path"]),
        "--device",
        DEVICE,
    ]
    if FAST_RUN:
        command.append("--fast")
    provenance = {
        "task": TASK,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "source_root": str(REPO),
        "source_revision": source_revision,
        "tracked_source_dirty": source_dirty,
        "source_sha256": source_files,
        "manifest_sha256": sha256_file(selected["manifest"]),
        "split_sha256": sha256_file(selected["split_path"]),
        "split_fingerprint": selected["fingerprint"],
        "config": selected["config"],
        "fast": FAST_RUN,
        "device": DEVICE,
        "python": sys.version,
        "command": command,
    }
    (experiment / "run_provenance.json").write_text(
        json.dumps(provenance, indent=2), encoding="utf-8"
    )
    env = os.environ.copy()
    env["PYTHONPATH"] = str(REPO)
    env["PYTHONUNBUFFERED"] = "1"
    print("Experiment:", experiment)
    with (experiment / "training.log").open("w", encoding="utf-8") as log:
        process = subprocess.Popen(
            command,
            cwd=REPO,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
        )
        try:
            for line in process.stdout:
                print(line, end="")
                log.write(line)
                log.flush()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            process.wait()
            raise
    (experiment / "run_status.json").write_text(
        json.dumps(
            {"returncode": returncode, "completed_utc": datetime.now(timezone.utc).isoformat()}
        ),
        encoding="utf-8",
    )
    RESULTS_RUN = str(experiment)
    if returncode:
        raise RuntimeError(f"Training failed ({returncode}); inspect {experiment / 'training.log'}")


## 6. View results

After training, this reads its saved summary. To inspect an earlier experiment, put its folder in `RESULTS_RUN` in step 1 and run this cell. Metrics describe the configured per-frame task; they do not establish full driver drowsiness detection. The test split is final evaluation; choose settings using training/validation results.

For remote runs, download the entire experiment folder before ending the session. Kaggle working storage is session storage until you explicitly save/download it. Keep checkpoints, config, provenance, splits, metrics, predictions and logs together.


In [ ]:
if RESULTS_RUN:
    result_dir = Path(RESULTS_RUN).expanduser().resolve()
    print("Experiment:", result_dir)
    for relative in ("run_status.json", "model/summary.json", "model/val.json", "model/test.json"):
        path = result_dir / relative
        if path.is_file():
            print(relative)
            print(json.dumps(json.loads(path.read_text(encoding="utf-8")), indent=2))
    if result_dir.is_dir():
        print("Saved files:")
        for path in sorted(result_dir.rglob("*")):
            if path.is_file():
                print(path.relative_to(result_dir), f"({path.stat().st_size:,} bytes)")
    else:
        print("Folder unavailable in this kernel. Copy or download it first.")
else:
    print("No experiment selected. Inspection-only execution produced no training results.")
